# Review map · Circumpolar AGBD trends with forest stand age and carbon accumulation
+ AGBD v3.1 OLS trends and Mann–Kendall trend classes, 2016–2025 (on at opening) and 2020–2025
+ Forest stand age, 2020 (ensemble mean, 30 m)
+ Carbon accumulation, 2020 (ensemble mean and uncertainty, 30 m)
+ CAVM treeline for context; click anywhere to query every layer at that pixel

Built with the shared Review map template (`maap_tools/lib/review_maps.py`) and the tile layers from
`icesat2_boreal/lib/mosaiclib.py`; saved as html next to the other published Review maps (`maap_tools/maps/`), so the
logos resolve and it can be viewed at `https://pahbs.github.io/maap_tools/maps/` after a push.

**Where to run (MAAP):** MAAP Hub or ADE with the `pangeo` image. Set the two repo paths in the next cell to where your
clones are.

In [ ]:
!pip install cogeo_mosaic==8.2.0

In [ ]:
import os, sys, json
import geopandas
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Repo clones used by this notebook (same paths as the Review map notebooks in maap_tools)
ICESAT2_BOREAL_LIB_PATH = '/projects/my-private-bucket/code/icesat2_boreal/lib'   # maplib_folium, mosaiclib
MAAP_TOOLS_PATH = '/projects/my-private-bucket/code/maap_tools'                  # lib/review_maps.py, maps/ (published)

sys.path.append(ICESAT2_BOREAL_LIB_PATH)
sys.path.append(os.path.join(MAAP_TOOLS_PATH, 'lib'))
import maplib_folium
import mosaiclib
import review_maps
from review_maps import WIP, COMPLETE

OUT_HTML = os.path.join(MAAP_TOOLS_PATH, 'maps', 'Review_maps_v3.1_AGBD_trends_age_cacc.html')

## AGBD v3.1 trends
**OLS trend:** per-pixel slope of the annual v3.1 AGBD maps (Mg/ha/yr), shown from −2 to +2 (PiYG: pink = loss, green =
gain). **Trend classes:** Mann–Kendall significance and strength of the trend, classes 1–10 (legend on the map). The
2016–2025 period is on at opening; 2020–2025 is in the layer list.

In [ ]:
TREND_PERIODS = ['2016-2025', '2020-2025']
TREND_LAYERS, TRENDCLASS_LAYERS = {}, {}
for i, PERIOD in enumerate(TREND_PERIODS):
    t = maplib_folium.make_tiles_layer_dict(
        None, mosaiclib.MISC_MOSAIC_JSON_FN_DICT[f'TRENDOLS_AGB_v3.1_multiyr_{PERIOD}'],
        f"AGBD v3.1 OLS trend {PERIOD} [Mg/ha/yr]",
        SHOW_CBAR=(i == 0),                      # one shared colorbar for the OLS trend layers
        PARAMS_DICT={"rescale": "-2,2", "bidx": "1", "colormap_name": 'PiYG'},
    )
    t['caption'] = "AGBD v3.1 OLS trend [Mg/ha/yr]"
    c = maplib_folium.make_tiles_layer_dict(
        None, mosaiclib.MISC_MOSAIC_JSON_FN_DICT[f'TRENDCLASS_AGB_v3.1_multiyr_{PERIOD}'],
        f"AGBD v3.1 trend classes {PERIOD}",
        SHOW_CBAR=False,                         # classes get a categorical legend (below), not a colorbar
        PARAMS_DICT={"rescale": "0,10", "bidx": "1", "colormap_name": 'RdBu_r'},
    )
    TREND_LAYERS[PERIOD], TRENDCLASS_LAYERS[PERIOD] = t, c
TREND_LAYERS['2016-2025']['layer'].show = True    # on at opening
TRENDS_LIST = [TREND_LAYERS[p] for p in TREND_PERIODS] + [TRENDCLASS_LAYERS[p] for p in TREND_PERIODS]

## Forest stand age and carbon accumulation, 2020
From the carbon accumulation product (`CACC_2020_v3.1_multiyr_nsims050`): band 1 = carbon accumulation mean, band 2 = its
standard deviation (MgC/ha/yr); band 9 = stand age mean, band 10 = its standard deviation (yr).

In [ ]:
CACC_MOSAIC = mosaiclib.MISC_MOSAIC_JSON_FN_DICT['CACC_2020_v3.1_multiyr_nsims050']
AGE_LAYER = maplib_folium.make_tiles_layer_dict(
    None, CACC_MOSAIC, "Forest stand age 2020 [yr] (ensemble mean, 30m)",
    SHOW_CBAR=True, PARAMS_DICT={"rescale": "1,250", "bidx": "9", "colormap_name": 'gist_earth'},
)
CACC_LAYER = maplib_folium.make_tiles_layer_dict(
    None, CACC_MOSAIC, "Carbon accumulation 2020 [MgC/ha/yr] (ensemble mean, 30m)",
    SHOW_CBAR=True, PARAMS_DICT={"rescale": "0,1", "bidx": "1", "colormap_name": 'turbo'},
)
CACC_SD_LAYER = maplib_folium.make_tiles_layer_dict(
    None, CACC_MOSAIC, "Carbon accumulation 2020 uncertainty [MgC/ha/yr] (ensemble st dev)",
    SHOW_CBAR=True, PARAMS_DICT={"rescale": "0,0.25", "bidx": "2", "colormap_name": 'magma'},
)
AGE_CACC_LIST = [AGE_LAYER, CACC_LAYER, CACC_SD_LAYER]

# Map

In [ ]:
boreal_tiles = geopandas.read_file(mosaiclib.boreal_tile_index_path)

# ESRI Light Gray is the only basemap on at opening
for k, bm in maplib_folium.basemaps.items():
    bm.show = (k == 'basemap_gray')
    bm._children.pop(bm.get_name() + '_add', None)   # clear the 'add to map' step left by an earlier display

MAP_LAYERS = AGE_CACC_LIST + TRENDS_LIST
# Every layer goes through MAP_REGISTERED_DPS_RESULTS, which adds the layer list and fullscreen button last;
# a layer added afterwards is referenced by the layer list before it exists, which breaks both controls.
m = maplib_folium.MAP_REGISTERED_DPS_RESULTS(
        boreal_tiles, None,
        CHECK_TILES_NAME='',
        map_width=750, map_height=500, SHOW_WIDGETS=False,
        ADD_TILELAYER=MAP_LAYERS,
        ADD_GEOJSONLAYER=[review_maps.cavm_treeline_layer()],   # dashed CAVM treeline, for context
    )
review_maps.drop_layer(m, 'Boreal tiles')
review_maps.set_extent(m, 'na_boreal')

### Trend class legend
Colours are sampled from the same colormap and range the tiles are drawn with (RdBu_r over 0–10), so the legend matches
the map. Class definitions from `icesat2_boreal/lib/compute_trends.py`.

In [ ]:
TREND_CLASS_LABELS = {
    1: "Strong sig. increase", 2: "Moderate sig. increase", 3: "Weak sig. increase", 4: "Very weak sig. increase",
    5: "Non-sig. (positive)", 6: "Non-sig. (negative)",
    7: "Very weak sig. decrease", 8: "Weak sig. decrease", 9: "Moderate sig. decrease", 10: "Strong sig. decrease",
}
cmap, norm = plt.get_cmap('RdBu_r'), mcolors.Normalize(0, 10)
rows = ''.join(
    f'<div style="display:flex;align-items:center;margin:1px 0;">'
    f'<span style="width:12px;height:12px;background:{mcolors.to_hex(cmap(norm(k)))};border:1px solid #666;'
    f'display:inline-block;margin-right:6px;flex-shrink:0;"></span><span>{k} · {v}</span></div>'
    for k, v in TREND_CLASS_LABELS.items())
legend_html = ('<div style="position:absolute;bottom:56px;left:12px;z-index:9999;background:rgba(255,255,255,0.92);'
               'padding:8px 10px;border:1px solid #9ca3af;border-radius:4px;font-family:sans-serif;font-size:11px;">'
               '<div style="font-weight:600;margin-bottom:4px;">AGBD trend classes (Mann&ndash;Kendall, &alpha;&nbsp;=&nbsp;0.05)</div>'
               + rows + review_maps.TREELINE_LEGEND_ROW + '</div>')
m.add_child(review_maps.FloatingLegend(legend_html))

## Pixel query
Click anywhere on the map: values of every layer at that pixel, including the uncertainty bands.

In [ ]:
query_layers = []
for PERIOD in TREND_PERIODS:
    query_layers += [
        {'section': f'AGBD v3.1 trend {PERIOD}', 'name': 'slope (OLS)', 'point_url': review_maps.point_url(TREND_LAYERS[PERIOD], bidx=1), 'units': 'Mg/ha/yr'},
        {'section': f'AGBD v3.1 trend {PERIOD}', 'name': 'p-value (Mann-Kendall)', 'point_url': review_maps.point_url(TREND_LAYERS[PERIOD], bidx=7), 'units': ''},
        {'section': f'AGBD v3.1 trend {PERIOD}', 'name': 'class', 'point_url': review_maps.point_url(TRENDCLASS_LAYERS[PERIOD], bidx=1), 'units': ''},
    ]
query_layers += [
    {'section': 'Forest stand age 2020', 'name': 'mean', 'point_url': review_maps.point_url(AGE_LAYER, bidx=9), 'units': 'yr'},
    {'section': 'Forest stand age 2020', 'name': 'st dev', 'point_url': review_maps.point_url(AGE_LAYER, bidx=10), 'units': 'yr'},
    {'section': 'Carbon accumulation 2020', 'name': 'mean', 'point_url': review_maps.point_url(CACC_LAYER, bidx=1), 'units': 'MgC/ha/yr'},
    {'section': 'Carbon accumulation 2020', 'name': 'st dev', 'point_url': review_maps.point_url(CACC_LAYER, bidx=2), 'units': 'MgC/ha/yr'},
]
m = maplib_folium.add_pixel_query(m, query_layers, decimals=2)
m

### Save to html

In [ ]:
PAGE = {
    'title': 'Circumpolar boreal AGBD trends, forest stand age and carbon accumulation',
    'subtitle': 'NASA Arctic/Boreal Vulnerability Experiment (ABoVE)',
    'logos': ['nasa', 'above', 'umd', 'edgehill', 'utaustin'],
    'contributors': ['montesano', 'duncanson', 'zarringhalam', 'minor', 'thomas', 'guenther', 'neuenschwander'],
    'note': 'Trends are per-pixel fits to the annual AGBD v3.1 maps (phase 1 OLS AGBD models from Duncanson et al. (2026) [1], '
            'phase 2 ICESat-2 &amp; HLS models updated using multi-year correspondence). Compare trends with stand age and '
            'carbon accumulation: young stands are expected to gain biomass, old stands to gain slowly or lose. Use the '
            'layer-control panel (top right) to toggle layers and basemaps; click anywhere on the map to query pixel values. '
            'Zoom in to display map tiles.',
    'refs': [review_maps.REF_DUNCANSON, review_maps.REF_MONTESANO_INPREP, review_maps.REF_GAMI],
    'rows': [
        {'layer': 'AGBD v3.1 OLS trend, 2016&ndash;2025 and 2020&ndash;2025', 'resolution': '30&thinsp;m',
         'description': 'Per-pixel OLS slope of annual v3.1 AGBD (Mg/ha/yr), shown from &minus;2 to +2 (PiYG). '
                        '2016&ndash;2025 on at opening. Not yet complete across the circumpolar domain.',
         'citation': 'Montesano et al. (in prep.) [2]; phase 1 AGBD models: Duncanson et al. (2026) [1]', 'status': WIP},
        {'layer': 'AGBD v3.1 trend classes, 2016&ndash;2025 and 2020&ndash;2025', 'resolution': '30&thinsp;m',
         'description': 'Mann&ndash;Kendall trend classes (&alpha;&thinsp;=&thinsp;0.05): 1&ndash;4 significant increases '
                        '(strong to very weak), 5&ndash;6 non-significant (positive, negative), 7&ndash;10 significant '
                        'decreases (very weak to strong); legend on the map.',
         'citation': 'Montesano et al. (in prep.) [2]; phase 1 AGBD models: Duncanson et al. (2026) [1]', 'status': WIP},
        {'layer': 'Forest stand age, 2020', 'resolution': '30&thinsp;m',
         'description': 'Ensemble mean stand age (yr), updated with TerraPulse and GAMI v2.0; shown from 1 to 250&thinsp;yr. '
                        'Pixel query adds the ensemble standard deviation.',
         'citation': 'Montesano et al. (in prep.) [2]; GAMI v2.0: Besnard et al. (2024) [3]', 'status': WIP},
        {'layer': 'Carbon accumulation, 2020', 'resolution': '30&thinsp;m',
         'description': 'Ensemble mean woody carbon accumulation rate (MgC/ha/yr), shown from 0 to 1; uncertainty layer '
                        '(ensemble standard deviation) shown from 0 to 0.25.',
         'citation': 'Montesano et al. (in prep.) [2]; GAMI v2.0: Besnard et al. (2024) [3]', 'status': WIP},
    ],
}
header_html = review_maps.build_header_html(PAGE['title'], PAGE['subtitle'], PAGE['rows'], PAGE['note'], PAGE['refs'],
                                            logos=PAGE['logos'], contributors=PAGE['contributors'])
review_maps.save_review_map(m, OUT_HTML, header_html)